# Fantasy Football Agent — LangGraph Orchestration Layer


In [4]:
!pip install -q chromadb openai requests
!pip install -q nfl_data_py --no-build-isolation
exec(open("/content/fantasy_agents.py").read())

  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (pyproject.toml) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
Mounted at /content/drive
ESPN fetcher defined.
[ChatCompletionMessageFunctionToolCall(id='call_dGQ6ah4qMs4bIDrH1afj0L4d', function=Function(arguments='{"table_names":["player_stats","snap_counts"]}', name='get_schemas'), type='function')]
Attempting SQL: SELECT player_display_name, SUM(carries + targets) as total_usage, SUM(carries) as total_carries, SUM(targets) as total_targets
FROM player_stats
WHERE season = 2024 AND position = 'RB'
GROUP BY player_display_name
ORDER BY total_usage DESC
L

In [ ]:
# @title Libraries
!pip install -q --force-reinstall langchain-core langchain-openai langgraph langchain-community
import os
import json
from typing import Annotated, Literal
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.types import Command

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

import operator
from google.colab import userdata

# ── Your existing sub-agent functions ──────────────────────────────────────
# These are imported from your original notebook cells.
# In Colab, just make sure those cells are run first.
# ask_agent(question: str) -> str          # SQL / historical stats
# ask_news_agent(question: str) -> str     # injury / news RAG
# ───────────────────────────────────────────────────────────────────────────


In [6]:
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## 1. Shared State

`AgentState` is the single object that flows through every node in the graph.
Each sub-agent writes its output into a dedicated field; the synthesizer reads all of them.

In [7]:
# @title Shared State definition

class AgentState(TypedDict):
    # The original user question
    question: str

    # Which sub-agents the orchestrator decided to invoke
    # e.g. ["stats", "news"] or ["stats"] or ["news"]
    agents_needed: list[str]

    # Sub-agent outputs — populated by each node, read by synthesizer
    stats_result: str     # historical stats / usage from nflverse
    news_result: str      # injury status / beat reporter notes
    matchup_result: str   # opponent defensive ratings
    vegas_result: str     # implied totals / game lines / props

    # Final answer assembled by the synthesizer
    final_answer: str

    completed_agents: Annotated[int, operator.add]

## 2. Orchestrator Node

The orchestrator's only job is **routing** — it reads the question and decides which
sub-agents are needed. It does not answer anything itself.

It uses `Command(goto=[...])` to fan out to one or more nodes simultaneously.

In [8]:
# @title Orchestrator node

llm = ChatOpenAI(model="gpt-4o", temperature=0)

ORCHESTRATOR_SYSTEM = """
You are the routing layer for a fantasy football analysis system.
Given a user question, decide which sub-agents are needed to answer it.

Available sub-agents:
- stats   : historical stats, snap counts, usage rates, season totals (nflverse DB)
- news    : current injury status, beat reporter updates, practice participation
- matchup : opponent defensive ratings, points allowed by position
- vegas   : game totals, implied team totals, player prop lines

Routing rules:
- Start/sit comparison      → stats, news, matchup, vegas
- Waiver wire pickup        → stats, news
- Pure historical question  → stats only
- Injury / availability     → news only
- Game environment / props  → vegas only (add stats/news for full player picture)
- Trade evaluation          → stats, news

Respond ONLY with a JSON object, no markdown, no explanation:
{"agents": ["stats", "news"]}   ← example
Valid agent names: stats, news, matchup, vegas
"""

def orchestrator_node(
    state: AgentState,
) -> Command[Literal["stats_agent", "news_agent", "matchup_agent", "vegas_agent"]]:
    """
    Reads the question, decides which sub-agents to call,
    and fans out to them in parallel via Command(goto=[...]).
    """
    response = llm.invoke([
        SystemMessage(content=ORCHESTRATOR_SYSTEM),
        HumanMessage(content=state["question"])
    ])

    try:
        routing       = json.loads(response.content)
        agents_needed = routing.get("agents", ["stats", "news", "matchup", "vegas"])
    except json.JSONDecodeError:
        agents_needed = ["stats", "news"]

    node_map = {
        "stats":   "stats_agent",
        "news":    "news_agent",
        "matchup": "matchup_agent",
        "vegas":   "vegas_agent",
    }
    target_nodes = [node_map[a] for a in agents_needed if a in node_map]

    print(f"[Orchestrator] Routing to: {target_nodes}")

    return Command(
        update={"agents_needed": agents_needed},
        goto=target_nodes
    )

## 3. Sub-Agent Nodes

Each node is a thin wrapper around the agent functions defined in `Fantasy_Agents.ipynb`.
They all share the same uniform structure: run the agent, write the result to state,
route to the synthesizer.

In [9]:
# @title Sub-agent nodes

def stats_agent_node(state: AgentState) -> Command[Literal["synthesizer"]]:
    print("[Stats Agent] Running...")
    try:
        result = ask_stats_agent(state["question"])
    except Exception as e:
        result = f"Stats agent failed: {str(e)}"
    return Command(
        update={"stats_result": result, "completed_agents": 1},
        goto="synthesizer"
    )


def news_agent_node(state: AgentState) -> Command[Literal["synthesizer"]]:
    print("[News Agent] Running...")
    try:
        result = ask_news_agent(state["question"])
    except Exception as e:
        result = f"News agent failed: {str(e)}"
    return Command(
        update={"news_result": result, "completed_agents": 1},
        goto="synthesizer"
    )


def matchup_agent_node(state: AgentState) -> Command[Literal["synthesizer"]]:
    print("[Matchup Agent] Running...")
    try:
        result = ask_matchup_agent(state["question"])
    except Exception as e:
        result = f"Matchup agent failed: {str(e)}"
    return Command(
        update={"matchup_result": result, "completed_agents": 1},
        goto="synthesizer"
    )


def vegas_agent_node(state: AgentState) -> Command[Literal["synthesizer"]]:
    print("[Vegas Agent] Running...")
    try:
        result = ask_vegas_agent(state["question"])
    except Exception as e:
        result = f"Vegas agent failed: {str(e)}"
    return Command(
        update={"vegas_result": result, "completed_agents": 1},
        goto="synthesizer"
    )

## 4. Synthesizer Node

The synthesizer collects all sub-agent outputs and produces the final fantasy recommendation.

In [20]:
# @title Synthesizer node

SYNTHESIZER_SYSTEM = """
You are a fantasy football analyst.

You will receive outputs from one or more specialist agents:
- Stats Agent: historical stats and usage data from the nflverse database
- News Agent: current injury status and beat reporter updates
- Matchup Agent: opponent defensive ratings and points allowed by position
- Vegas Agent: implied team totals and game lines

Your job:
1. Synthesize all available signals into a coherent fantasy analysis.
2. Identify conflicts (e.g., great historical stats but injury concern) and call them out.
3. Produce a clear recommendation with a confidence level: High / Medium / Low.
4. Keep it concise — a fantasy manager needs to make a decision, not read a dissertation.

Format your response as:
**Confidence:** [High / Medium / Low]
**Key factors:**
- <factor 1>
- <factor 2>
- ...
**Analysis:** [2-5 sentence narrative]
**Caveats:** [anything that could flip this recommendation]

"""

def synthesizer_node(state: AgentState) -> Command[Literal["synthesizer", END]]:
    # Wait until all dispatched agents have reported in
    if state.get("completed_agents", 0) < len(state.get("agents_needed", [])):
        return Command(goto="synthesizer")  # not ready yet, re-enter

    print("[Synthesizer] All agents complete — combining signals...")

    sections = []
    if state.get("stats_result"):
        sections.append(f"=== HISTORICAL STATS (Stats Agent) ===\n{state['stats_result']}")
    if state.get("news_result"):
        sections.append(f"=== INJURY / NEWS (News Agent) ===\n{state['news_result']}")
    if state.get("matchup_result"):
        sections.append(f"=== MATCHUP DATA (Matchup Agent) ===\n{state['matchup_result']}")
    if state.get("vegas_result"):
        sections.append(f"=== VEGAS / ODDS (Vegas Agent) ===\n{state['vegas_result']}")

    if not sections:
        return Command(
            update={"final_answer": "No agent data returned."},
            goto=END
        )

    combined_context = "\n\n".join(sections)
    user_message = f"Original question: {state['question']}\n\nAgent outputs:\n{combined_context}"

    response = llm.invoke([
        SystemMessage(content=SYNTHESIZER_SYSTEM),
        HumanMessage(content=user_message)
    ])

    return Command(
        update={"final_answer": response.content},
        goto=END
    )

In [21]:
# @title Build and compile the graph

builder = StateGraph(AgentState)

# Register all nodes
builder.add_node("orchestrator",  orchestrator_node)
builder.add_node("stats_agent",   stats_agent_node)
builder.add_node("news_agent",    news_agent_node)
builder.add_node("matchup_agent", matchup_agent_node)
builder.add_node("vegas_agent",   vegas_agent_node)
builder.add_node("synthesizer",   synthesizer_node)

# Entry point
builder.add_edge(START, "orchestrator")

# Orchestrator → sub-agents: handled dynamically by Command(goto=[...]) inside the node.
# No explicit edges needed

# Compile
fantasy_graph = builder.compile()

print("Graph compiled successfully.")
print("Nodes:", list(fantasy_graph.get_graph().nodes.keys()))

Graph compiled successfully.
Nodes: ['__start__', 'orchestrator', 'stats_agent', 'news_agent', 'matchup_agent', 'vegas_agent', 'synthesizer', '__end__']


In [22]:
# @title Main entry point

def analyze(question: str, verbose: bool = True) -> str:
    """
    Run the full fantasy agent graph for a given question.

    Args:
        question: Natural language fantasy football question.
        verbose:  If True, prints routing and agent status as nodes execute.

    Returns:
        Final synthesized recommendation as a string.
    """
    # Initialize state — all result fields start empty
    initial_state: AgentState = {
        "question":       question,
        "agents_needed":  [],
        "stats_result":     "",
        "news_result":    "",
        "matchup_result": "",
        "vegas_result":   "",
        "final_answer":   "",
        "completed_agents":  0
    }

    if verbose:
        print(f"\n{'='*60}")
        print(f"Question: {question}")
        print('='*60)

    final_state = fantasy_graph.invoke(initial_state)

    if verbose:
        print(f"\nAgents used: {final_state['agents_needed']}")
        print(f"{'='*60}\n")

    return final_state["final_answer"]

In [25]:
analyze("Which running backs had the highest yardage in the 2024? Give carries and targets, sort by the summation, limit 10?")


Question: Which running backs had the highest yardage in the 2024? Give carries and targets, sort by the summation, limit 10?
[Orchestrator] Routing to: ['stats_agent']
[Stats Agent] Running...
[ChatCompletionMessageFunctionToolCall(id='call_8SCISRKuVZ9vMpR2IATJOBNb', function=Function(arguments='{"table_names":["player_stats","snap_counts"]}', name='get_schemas'), type='function')]
Attempting SQL: SELECT player_display_name, SUM(rushing_yards + receiving_yards) as total_yards, SUM(carries) as total_carries, SUM(targets) as total_targets FROM player_stats WHERE position = 'RB' AND season = 2024 GROUP BY player_display_name ORDER BY total_yards DESC LIMIT 10;
  player_display_name  total_yards  total_carries  total_targets
0      Saquon Barkley       2857.0          436.0           58.0
1       Derrick Henry       2384.0          367.0           24.0
2        Jahmyr Gibbs       2084.0          264.0           74.0
3      Bijan Robinson       1887.0          304.0           72.0
4      

"**Confidence:** High  \n**Key factors:**  \n- Saquon Barkley led with 2,857 total yards, significantly ahead of others.  \n- Derrick Henry and Jahmyr Gibbs also had strong yardage, with Henry focusing more on carries and Gibbs on targets.  \n- Bijan Robinson and Josh Jacobs rounded out the top five with balanced usage.  \n\n**Analysis:** Saquon Barkley was the standout running back in 2024, with a massive workload of 436 carries and 58 targets, leading to a league-high 2,857 total yards. Derrick Henry and Jahmyr Gibbs followed, with Henry's production heavily reliant on his 367 carries, while Gibbs balanced his yardage with 74 targets. Bijan Robinson and Josh Jacobs also delivered strong performances, showcasing versatility in both the running and passing game. These players were key contributors to their teams and fantasy rosters alike.  \n\n**Caveats:** Injuries or changes in team dynamics could impact future performance. Additionally, the reliance on high carry counts for players l